# From Voice to Vision — 6. Robust Optimisation and Ensemble

The final optimisation protocol adds two safeguards to the enhanced pipeline.

The population of each algorithm is **seeded** with a known-good default configuration, so the
search can only refine an already reasonable solution rather than wander into fast-but-weak regions
of the space, and the search bounds are restricted to a sensible regime.

Once the search terminates, the default configuration and the winner of each algorithm are all
**retrained to convergence**. The best single model is selected on the validation set, and all four
are additionally combined into a **soft-voting ensemble** that averages their predicted class
probabilities.

In [ ]:
# Clone the project repository and install the audio dependencies
REPO_URL = "https://github.com/Nadaa3672/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

In [ ]:
from src import config, data_loader, features
import numpy as np
data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)
data["split"] = df["split"].to_numpy()
splits = features.split_arrays(data)
print("Split sizes:", {s: len(splits[s]["y"]) for s in ["train", "val", "test"]})

In [ ]:
QUICK = False
if QUICK:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 4, 2, 10
else:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 5, 4, 18

from src.optimization import pso, fso, ga
from src.optimization.search_space import make_objective, decode, default_u, DEFAULT_HP, DIM
import time

SEED_U = default_u()   # the default configuration, used to seed each population

def run(optimizer, **kw):
    obj = make_objective(splits, epochs=EPOCHS_SEARCH, patience=5)
    t0 = time.time()
    res = optimizer.optimize(obj, DIM, seed=config.SEED, seed_u=SEED_U, **kw)
    res["time_s"] = time.time() - t0
    res["n_evals"] = len(obj.history_evals)
    res["best_hp"] = decode(res["best_u"])
    print(f"{res['name']}: robust validation = {res['best_fit']:.3f} | "
          f"evaluations = {res['n_evals']} | time = {res['time_s']:.0f} s")
    return res

In [ ]:
res_pso = run(pso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_pso["best_hp"])

In [ ]:
res_fso = run(fso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_fso["best_hp"])

In [ ]:
res_ga = run(ga, pop_size=N_AGENTS, n_iter=N_ITER); print(res_ga["best_hp"])

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
allres = [res_pso, res_fso, res_ga]
plt.figure(figsize=(9, 5))
for r in allres:
    plt.plot(range(len(r["history"])), r["history"], marker="o", label=r["name"])
plt.xlabel("iteration"); plt.ylabel("robust validation accuracy")
plt.title("Convergence with seeded populations"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "05_convergence.png", dpi=150)
plt.show()

display(pd.DataFrame([{"algorithm": r["name"], "robust_val": round(r["best_fit"], 3),
                       "evaluations": r["n_evals"], "time_s": round(r["time_s"]),
                       **r["best_hp"]} for r in allres]))

## Retraining the finalists

Each finalist is retrained to convergence. The best single model is chosen on validation accuracy,
without inspecting the test set.

In [ ]:
from src.models import cnn
finalists = {"default": DEFAULT_HP, "PSO": res_pso["best_hp"],
             "FSO": res_fso["best_hp"], "GA": res_ga["best_hp"]}
trained = {}
for name, hp in finalists.items():
    out = cnn.train_cnn(splits, hp, epochs=60, patience=12,
                        deltas=True, augment=True, verbose=False)
    trained[name] = out
    print(f"  {name:8s}: validation = {out['best_val_acc']:.3f} | test = {out['test_acc']:.3f}")

best_name = max(trained, key=lambda k: trained[k]["best_val_acc"])
best = trained[best_name]
print(f"\nBest single model by validation: {best_name} → test = {best['test_acc']:.3f}")

## Soft-voting ensemble

Averaging the class probabilities of the four finalists exploits the diversity of the optima found
by the different algorithms.

In [ ]:
models = [trained[k]["model"] for k in trained]
ens_acc, ens_probs, y_true = cnn.ensemble_evaluate(models, splits, split_name="test")
print(f"Ensemble test accuracy = {ens_acc:.3f}")

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
ens_pred = ens_probs.argmax(1)
print(classification_report(y_true, ens_pred, target_names=config.EMOTIONS, zero_division=0))

plt.figure(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_true, ens_pred), annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=config.EMOTIONS, yticklabels=config.EMOTIONS)
plt.title(f"Ensemble — test = {ens_acc:.2f}")
plt.xlabel("predicted"); plt.ylabel("true"); plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "05_confusion_ensemble.png", dpi=150)
plt.show()

In [ ]:
improv = pd.DataFrame([
    {"model": "SVM (MFCC)", "test_acc": 0.517},
    {"model": "CNN baseline (no augmentation)", "test_acc": 0.529},
    {"model": "CNN enhanced (default hp)", "test_acc": round(trained["default"]["test_acc"], 3)},
    {"model": f"CNN optimised ({best_name})", "test_acc": round(best["test_acc"], 3)},
    {"model": "Ensemble (PSO+FSO+GA+default)", "test_acc": round(ens_acc, 3)},
])
plt.figure(figsize=(9, 4.5))
sns.barplot(data=improv, x="test_acc", y="model", color="#4C78A8")
for i, v in enumerate(improv["test_acc"]):
    plt.text(v + 0.01, i, f"{v:.3f}", va="center")
plt.xlim(0, 1); plt.xlabel("Test accuracy (speaker-independent)")
plt.title("Accuracy progression across the pipeline")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "05_progression.png", dpi=150)
plt.show()
display(improv)

import json, torch
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "best_hp.json", "w") as f:
    json.dump({"winner": best_name, "hp": finalists[best_name],
               "val_acc": best["best_val_acc"], "test_acc": best["test_acc"],
               "ensemble_test_acc": ens_acc}, f, indent=2)
with open(config.RESULTS_DIR / "optimization_results.json", "w") as f:
    json.dump([{"name": r["name"], "robust_val": r["best_fit"], "n_evals": r["n_evals"],
                "time_s": r["time_s"], "history": r["history"], "hp": r["best_hp"]}
               for r in allres], f, indent=2)
torch.save({"state_dict": best["model"].state_dict(), "hp": finalists[best_name],
            "norm": best["norm"], "in_channels": best["in_channels"]},
           config.RESULTS_DIR / "best_cnn.pt")

The ensemble attains the best result of the study. Notably, the configuration discovered by the
Flock-of-Starlings Optimizer yields the strongest *single* model on the test set, even though
selection strictly by validation returns the seeded default — an honest illustration of the
imperfect correlation between validation and test on a corpus of this size, and a further argument
in favour of combining diverse optima rather than trusting a single one.